In [ ]:
import os
import glob
import json
import pandas as pd

# Directories
DATA_DIR = os.path.join("..", "data", "model_ready")
ARTIFACTS_DIR = os.path.join("..", "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# The GEOHASHes will be collected and a mapping created (All the data won't be loaded!)
parquet_files = glob.glob(os.path.join(DATA_DIR, "*.parquet"))
all_geohashes = set()

print(f"[*] {len(parquet_files)} Searching for GEOHASHes...")
for f in parquet_files:
    # Reading only GEOHASH columns
    temp_df = pd.read_parquet(f, columns=['GEOHASH'])
    all_geohashes.update(temp_df['GEOHASH'].unique())
    print(f"[+] Scanned: {os.path.basename(f)}", end='\r')

# Saving the mapping file
geohash_to_id = {val: idx for idx, val in enumerate(sorted(list(all_geohashes)))}
mapping_path = os.path.join(ARTIFACTS_DIR, "geohash_mapping.json")

with open(mapping_path, "w", encoding="utf-8") as f:
    json.dump(geohash_to_id, f)

print(f"\n[+] Mapping complete! {len(geohash_to_id)} unique grid found.")
print(f"[+] file saved: {mapping_path}")

[*] 61 dosyadan Geohash'ler taranıyor...
[+] Taranan: trafik_verisi_2025_01.parquet
[+] Mapping tamamlandı! 5014 adet benzersiz bölge bulundu.
[+] Dosya kaydedildi: ..\artifacts\geohash_mapping.json


In [ ]:
import xgboost as xgb
import json
import pandas as pd
import os
import glob
from tqdm import tqdm

# Settings
DATA_DIR = os.path.join("..", "data", "model_ready")
ARTIFACTS_DIR = os.path.join("..", "artifacts")
MAPPING_PATH = os.path.join(ARTIFACTS_DIR, "geohash_mapping.json")

# load the mapping file
with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    geohash_to_id = json.load(f)

# Model commence!
model = xgb.XGBRegressor(n_estimators=50, learning_rate=0.1, tree_method='hist')
model_fitted = False

parquet_files = glob.glob(os.path.join(DATA_DIR, "*.parquet"))

print("[*] Training in process. Files will be processed with a text-based progress bar...")

# tqdm
for f in tqdm(parquet_files, desc="Model is being trained..."):
    df = pd.read_parquet(f)
    df['geohash_id'] = df['GEOHASH'].map(geohash_to_id)
    
    # Selecting features
    X = df[['geohash_id', 'hour', 'dayOfWeek', 'isHoliday', 'temp', 'precip', 'wind']]
    y = df['AVERAGE_SPEED']
    
    # Training
    if not model_fitted:
        model.fit(X, y)
        model_fitted = True
    else:
        model.fit(X, y, xgb_model=model)

print("[+] Training successful!")

# Save
model_path = os.path.join(ARTIFACTS_DIR, "traffic_model.json")
model.save_model(model_path)
print(f"[+] Model saved -> {model_path}")

[*] Eğitim başlıyor. Dosyalar metin tabanlı ilerleme çubuğuyla işlenecek...


Model Eğitiliyor: 100%|██████████| 61/61 [15:34<00:00, 15.31s/it]


[+] Eğitim başarıyla tamamlandı!
[+] Model başarıyla kaydedildi -> ..\artifacts\traffic_model.json


In [ ]:
import os
import xgboost as xgb
import onnxmltools
from onnxconverter_common.data_types import FloatTensorType

ARTIFACTS_DIR = os.path.join("..", "artifacts")
model_json_path = os.path.join(ARTIFACTS_DIR, "traffic_model.json")

# load the model
loaded_model = xgb.XGBRegressor()
loaded_model.load_model(model_json_path)

# updating the feature names as f0, f1, f2,... due to the fact
# that onnxmltools does not support string names.
booster = loaded_model.get_booster()
booster.feature_names = [f"f{i}" for i in range(7)]  # 7 parameters

# ONNX transformation setup
initial_types = [('float_input', FloatTensorType([None, 7]))]

print("[*] Model is being transformed into ONNX format...")
onnx_model = onnxmltools.convert_xgboost(loaded_model, initial_types=initial_types)

# saving into artifacts
onnx_path = os.path.join(ARTIFACTS_DIR, "traffic_model.onnx")
onnxmltools.utils.save_model(onnx_model, onnx_path)

print(f"[+] Success! ONNX model has been saved -> {onnx_path}")

[*] Model ONNX formatına dönüştürülüyor...
[+] Başarılı! ONNX modeli kaydedildi -> ..\artifacts\traffic_model.onnx
